In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

In [ ]:
(mnist_x, _), _ = mnist.load_data()

target_patch = np.array([
    [1, 154, 253, 90],
    [0, 139, 253, 190],
    [0, 11, 190, 253],
    [0, 0, 35, 241]
], dtype=np.int32)

input_patch = None
for img in mnist_x:
    for r in range(img.shape[0] - 3):
        for c in range(img.shape[1] - 3):
            subgrid = img[r:r+4, c:c+4]
            if np.array_equal(subgrid, target_patch):
                input_patch = subgrid
                break
        if input_patch is not None:
            break
    if input_patch is not None:
        break

if input_patch is None:
    input_patch = target_patch

In [ ]:
kernel = np.array([
    [1, 0],
    [0, -1]
], dtype=np.int32)

def convolve2d(image, kernel):
    i_h, i_w = image.shape
    k_h, k_w = kernel.shape
    o_h = i_h - k_h + 1
    o_w = i_w - k_w + 1
    output = np.zeros((o_h, o_w), dtype=np.int32)
    for r in range(o_h):
        for c in range(o_w):
            output[r, c] = np.sum(image[r:r+k_h, c:c+k_w] * kernel)
    return output

def maxpool2d(image, pool_size=2, stride=2):
    i_h, i_w = image.shape
    o_h = (i_h - pool_size) // stride + 1
    o_w = (i_w - pool_size) // stride + 1
    output = np.zeros((o_h, o_w), dtype=np.int32)
    for r in range(o_h):
        for c in range(o_w):
            r_start = r * stride
            c_start = c * stride
            output[r, c] = np.max(image[r_start:r_start+pool_size, c_start:c_start+pool_size])
    return output

feature_map = convolve2d(input_patch, kernel)
pooled_output = maxpool2d(feature_map, pool_size=2, stride=2)

In [ ]:
print("Input Patch:")
print(input_patch)
print("\nFeature Map (After Convolution):")
print(feature_map)
print(f"Shape: {feature_map.shape}")
print("\nPooled Output:")
print(pooled_output)
print(f"Shape: {pooled_output.shape}")